# Multisample MEBOCOST analysis for neck adipose tissue

This notebook is the main **multicondition MEBOCOST workflow** for the neck adipose dataset.

It performs four linked tasks:

1. Load the full neck adipose AnnData object.
2. Run MEBOCOST across both neck regions in one object using `neck_region` as the condition column.
3. Export the average expression matrix required by **COMPASS**.
4. Re-import the COMPASS-constrained communication results, run differential communication analysis, and export the final communication tables used downstream in R.

## Important downstream handoff

The most important downstream file for the later R analysis is:

- `communication_result_multi.csv`

This notebook also saves several important intermediate files:

- `./data/NeckAdipose/NeckAdipose_commu_multi.pk`
- `avg_exp_mat_multi.tsv`
- `results_commu_multi.csv`

Those should all be preserved because they support rerunning or extending the MEBOCOST workflow.


In [15]:
# -------------------------------------------------------------------------
# Load required packages
# -------------------------------------------------------------------------
import os
import sys
from datetime import datetime
from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

from mebocost import mebocost

# -------------------------------------------------------------------------
# Define inputs, outputs, and analysis settings
# -------------------------------------------------------------------------

PROJECT_ROOT = Path("Projects/Camara_2025_snRNAseq/NeckAdipose") #Adjust this path to your project root directory

INPUT_H5AD = PROJECT_ROOT / "data" / "adata_integrated_soupXoutput_with_infer_forR_temp_Dec18_cleaned.h5ad"
INPUT_H5AD = "data/NeckAdipose/adata_integrated_soupXoutput_with_infer_forR_temp_Dec18_cleaned.h5ad"
CONFIG_PATH = PROJECT_ROOT / "scripts" / "4.downstream_pipelines" / "mebocost" / "mebocost.conf"

MEBO_OUTPUT_DIR = PROJECT_ROOT / "output" / "4.downstream_pipelines" / "mebocost" / "data"
os.makedirs(MEBO_OUTPUT_DIR, exist_ok=True)

COMPASS_OUTPUT_DIR = PROJECT_ROOT / "output" / "4.downstream_pipelines" / "compass" / "data"
os.makedirs(COMPASS_OUTPUT_DIR, exist_ok=True)

OUTPUT_MEBO_OBJ = MEBO_OUTPUT_DIR / "NeckAdipose_commu_multi.pk"
OUTPUT_AVG_EXP = MEBO_OUTPUT_DIR / "avg_exp_mat_multi.tsv"
OUTPUT_DIFF_RES = MEBO_OUTPUT_DIR / "results_commu_multi.csv"
OUTPUT_COMM_RES = MEBO_OUTPUT_DIR / "communication_result_multi.csv"

COMPASS_FOLDER = COMPASS_OUTPUT_DIR / "compass_res_multi_Deep_Sup"

GROUP_COL = "cell_type"
CONDITION_COL = "neck_region"
SPECIES = "human"
THREADS = 8
MIN_CELL_NUMBER = 10
N_SHUFFLE = 1000
SEED = 12345


## 1. Load the multicondition AnnData object

The AnnData object must contain:
- gene expression values
- `cell_type` in `adata.obs`
- `neck_region` in `adata.obs`

These are required because `cell_type` defines sender/receiver groups and
`neck_region` defines the conditions for differential communication analysis.


In [5]:
adata = sc.read_h5ad(INPUT_H5AD)

print(f"# of cell: {adata.shape[0]}, # of gene: {adata.shape[1]}")
print("Conditions:", adata.obs[CONDITION_COL].unique().tolist())

# If needed in another dataset, one could restore all genes from .raw:
# adata = adata.raw.copy()


# of cell: 37596, # of gene: 32981
Conditions: ['Deep', 'Intermediate', 'Superficial']


## 2. Create the MEBOCOST object

This step initializes the multicondition MEBOCOST object directly from the
AnnData object. The analysis is grouped by `cell_type` and stratified by
`neck_region`.


In [22]:
mebo_obj = mebocost.create_obj(
                        adata = adata,
                        ## "celltype" must be a column name of adata.obs table, otherwise, change it according to your data.
                        group_col = GROUP_COL,
                        ## "Site" must be a column name of adata.obs table, otherwise, change it according to your data.
                        condition_col = CONDITION_COL,
                        met_est = 'mebocost',
                        # make sure mebocost.conf file in the same folder of this notebook, otherwise, provide a absolute path.
                        config_path = CONFIG_PATH, 
                        exp_mat=None,
                        cell_ann=None,
                        ## make sure you set the right species
                        species=SPECIES,
                        met_pred=None,
                        met_enzyme=None,
                        met_sensor=None,
                        met_ann=None,
                        scFEA_ann=None,
                        compass_met_ann=None,
                        compass_rxn_ann=None,
                        cutoff_exp='auto', ## automated cutoff to exclude lowly ranked 25% sensors across all cells
                        cutoff_met='auto', ## automated cutoff to exclude lowly ranked 25% metabolites across all cells
                        cutoff_prop=0.15, ## at lease 10% of cells should be expressed the sensor or present the metabolite in the cell group (specified by group_col)
                        sensor_type='All',
                        thread=THREADS
                        )


[April 20, 2026 16:00:32]: We get expression data with 32981 genes and 37596 cells.
[April 20, 2026 16:00:32]: Data Preparation Done in 0.1720 seconds


## 3. Infer communication events across conditions

This is the main MEBOCOST communication inference step.

Important detail:
- `save_permuation=True` is kept because the notebook later performs differential communication analysis.


In [23]:
## metabolic communication inference, this step takes a while
commu_res = mebo_obj.infer_commu(
                                n_shuffle=N_SHUFFLE,
                                seed=SEED, 
                                Return=True, 
                                thread=None,
                                ### be sure to set to True if you plan to do differential analysis
                                save_permuation=True,
                                ## change it based on your own data to filter out cell type with too few cell numbers
                                min_cell_number = MIN_CELL_NUMBER,
                                pval_method='permutation_test_fdr',
                                pval_cutoff=0.05
                            )

print("Number of significant mCCC detected by enzyme and sensor co-expression:", commu_res.shape[0])
print("Number of significant mCCC by condition:")
print(commu_res.groupby("Condition").size())
commu_res.head()


[April 20, 2026 16:00:36]: Load config and read data based on given species [human].
[April 20, 2026 16:00:39]: Estimtate metabolite enzyme expression using mebocost
[April 20, 2026 16:00:50]: Infer communications
[April 20, 2026 16:00:50]: Sensor type used ['Transporter', 'Receptor; Transporter', 'Nuclear Receptor', 'Receptor', 'Receptor; Channel', 'Channel', 'Enzyme']
[April 20, 2026 16:00:50]: Parameters: {shuffling: 1000 times, random seed: 12345, thread: 8}
[April 20, 2026 16:01:05]: met_sensor: (594, 8)
[April 20, 2026 16:01:05]: avg_exp: (2332, 53) for (gene, cell) of needed
[April 20, 2026 16:01:05]: avg_met: (591, 53) for (metabolite, cell) of needed
[April 20, 2026 16:01:05]: shuffling 1000 times for generating backgroud
[April 20, 2026 16:01:38]: take exp and met avg for shuffling
[April 20, 2026 16:02:13]: thread: 8
[April 20, 2026 16:02:17]: ABCA1 ~ HMDB0006247
[April 20, 2026 16:02:17]: Calculating P-value
[April 20, 2026 16:02:18]: ADORA1 ~ HMDB0000045
[April 20, 2026 16

,Sender,Receiver,Condition,Metabolite,Metabolite_Name,Sensor,Annotation,Commu_Score,Norm_Commu_Score,met_in_sender,sensor_in_receiver,metabolite_prop_in_sender,sensor_prop_in_receiver,ttest_stat,ttest_pval,permutation_test_stat,permutation_test_pval,ttest_fdr,permutation_test_fdr
489,Intermediate ~ Lymphatic Endothelial Cells,Intermediate ~ Pre-adipocytes,Intermediate,HMDB0000234,Testosterone,AR,Nuclear Receptor,0.042589,0.851781,0.069954,0.608815,0.156863,0.203077,-8.864269,1.736306e-18,0.0,0.0,8.695291e-17,0.0
1859,Deep ~ Pericytes,Deep ~ Adipocyte Progenitor Cells,Deep,HMDB0000097,Choline,SLC44A1,Transporter,0.050401,1.008010,0.110924,0.454370,0.425532,0.190067,-5.817387,4.021276e-09,0.0,0.0,1.183666e-07,0.0
2058,Deep ~ B Lymphocytes,Deep ~ Parathyroid Associated Cells,Deep,HMDB0000097,Choline,SLC44A1,Transporter,0.050401,1.008014,0.072451,0.695656,0.285714,0.266075,-3.484464,2.572629e-04,0.0,0.0,4.033119e-03,0.0
1686,Superficial ~ Pre-adipocytes,Superficial ~ Schwann Cells,Superficial,HMDB0000097,Choline,SLC44A1,Transporter,0.050501,1.010012,0.055043,0.917473,0.363823,0.346939,-11.157276,1.258849e-27,0.0,0.0,8.554884e-26,0.0
708,Intermediate ~ Smooth Muscle Cells,Intermediate ~ Schwann Cells,Intermediate,HMDB0000097,Choline,SLC44A1,Transporter,0.051499,1.029982,0.062151,0.828612,0.311813,0.311688,-11.963406,3.194525e-31,0.0,0.0,2.381477e-29,0.0


## 4. Save and reload the multicondition MEBOCOST object

The saved pickle object is an important intermediate because it allows
the downstream COMPASS-constrained and differential analysis steps to be rerun
without recreating the object from scratch.


In [13]:
mebocost.save_obj(obj=mebo_obj, path=OUTPUT_MEBO_OBJ)


In [14]:
# Reload when needed
mebo_obj = mebocost.load_obj(OUTPUT_MEBO_OBJ)

print("sensor_exp cutoff:", mebo_obj.cutoff_exp)
print("metabolite_agg_enzyme cutoff:", mebo_obj.cutoff_met)


[April 20, 2026 15:52:13]: Data Preparation Done in 0.0716 seconds
sensor_exp cutoff: 1.3434919118881226
metabolite_agg_enzyme cutoff: 0.16674520903163484


## 5. Export average expression matrix for COMPASS

COMPASS is run **between** the main MEBOCOST inference and the final
differential communication analysis.

This exported matrix is one of the key MEBOCOST intermediates:
- `avg_exp_mat_multi.tsv`

It contains average expression by `neck_region` and `cell_type`, and is
used as input to COMPASS.


In [16]:
avg_exp = sc.get.aggregate(adata, by=[CONDITION_COL, GROUP_COL], func="mean")

avg_exp = pd.DataFrame(
    avg_exp.layers["mean"],
    index=avg_exp.obs[CONDITION_COL].astype(str) + " ~ " + avg_exp.obs[GROUP_COL].astype(str),
    columns=avg_exp.var_names
).T

# Undo log transform because COMPASS performs its own internal log handling
avg_exp = avg_exp.apply(lambda col: np.exp(col) - 1)

avg_exp.to_csv(OUTPUT_AVG_EXP, sep="\t")


## 6. Constrain MEBOCOST communication scores with COMPASS flux results

After COMPASS is run externally on `avg_exp_mat_multi.tsv`, its secretion and
uptake outputs can be used to constrain MEBOCOST communication events.

This step updates `mebo_obj.commu_res` to reflect the flux-constrained result.


In [17]:
updated_res = mebo_obj._ConstrainCompassFlux_(
    compass_folder=COMPASS_FOLDER,
    efflux_cut="auto",
    influx_cut="auto",
    inplace=False
)

mebo_obj.commu_res = updated_res

print("Number of mCCC detected after flux constraints:")
print(updated_res.groupby("Condition").size())


efflux_cut: 53.38539126015655
influx_cut: 2.387013265166879
Number of mCCC detected after flux constraints:
Condition
Deep            1265
Intermediate     633
Superficial      595
dtype: int64


## 7. Differential communication analysis

This section compares communication events between Deep and Superficial
neck regions.

The key differential output saved here is:

- `differential_communication_result_multi.csv`


In [18]:
print("Available conditions:", mebo_obj.cell_ann[mebo_obj.condition_col].unique().tolist())


Available conditions: ['Deep', 'Intermediate', 'Superficial']


In [19]:
comp_cond = ["Deep_vs_Superficial"]

mebo_obj.CommDiff(
    comps=comp_cond,
    sig_mccc_only=True,
    flux_pass=True,
    thread=THREADS
)


[April 20, 2026 15:52:36]: Diff Comm for Deep_vs_Superficial
[April 20, 2026 15:52:36]: Compute diff mCCC
[April 20, 2026 15:52:36]: Cond1 is Deep, Cond2 is Superficial
[April 20, 2026 15:52:36]: Compute fold change
[April 20, 2026 15:52:46]: Estimate background diff
[April 20, 2026 15:54:04]: Test significance
[April 20, 2026 15:54:16]: Diff mCCC Analysis Done in 99.4676 seconds
[April 20, 2026 15:54:16]: Memory Usage in Peak 0.72 GB


In [20]:
print("Differential result is ready for:", list(mebo_obj.diffcomm_res.keys()))

comp = "Deep_vs_Superficial"
diff_res = mebo_obj.diffcomm_res[comp].copy()

# Save the differential communication table
mebo_obj.diffcomm_res[comp].to_csv(OUTPUT_DIFF_RES, index=False)

# Save the updated MEBOCOST object including differential results
mebocost.save_obj(obj=mebo_obj, path=OUTPUT_MEBO_OBJ)


Differential result is ready for: ['Deep_vs_Superficial']


## 8. Export the full communication result used downstream in R

This is the most important downstream handoff file for later R analysis:

- `communication_result_multi.csv`

It is the tidy communication table extracted from the updated MEBOCOST object.


In [21]:
commu_res = mebo_obj.commu_res.copy()

# Optional FDR filtering could be applied here if desired:
# commu_res = commu_res[commu_res["permutation_test_fdr"] <= 0.05]

commu_res.to_csv(OUTPUT_COMM_RES, sep=",", index=None)
